<a href="https://colab.research.google.com/github/dhruvsuri8106-code/ECON-3916---Statistical-and-Machine-Learning/blob/main/Lab%2012%20/%20Lab_12.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [2]:
import pandas as pd
import numpy as np
import statsmodels.formula.api as smf
from statsmodels.tools.eval_measures import rmse
import matplotlib.pyplot as plt

# Step 1: Ingestion from external source
url = 'https://raw.githubusercontent.com/epleywin/ECON3916-Statistical-Machine-Learning/refs/heads/main/Data/Zillow_ZHVI_2026_Micro.csv'
df = pd.read_csv(url)

In [5]:
# ============================================================
# RESIDUAL FORENSICS DASHBOARD
# Hedonic Pricing OLS Model — Statsmodels + Plotly Express
# ============================================================

import numpy as np
import pandas as pd
import statsmodels.api as sm
import plotly.express as px
import plotly.graph_objects as go

# ------------------------------------------------------------
# SECTION 1: EXAMPLE DATA + MODEL (replace with your own)
# ------------------------------------------------------------
# Swap this block out for your own DataFrame and formula.
# This synthetic example mimics a hedonic housing price model.

np.random.seed(42)
n = 300

df = pd.DataFrame({
    "sqft":        np.random.randint(500, 4000, n),
    "bedrooms":    np.random.randint(1, 6, n),
    "bathrooms":   np.random.randint(1, 4, n),
    "age_years":   np.random.randint(0, 60, n),
    "dist_cbd_km": np.random.uniform(1, 30, n),
})

# Simulate log-linear hedonic price (common in real estate econometrics)
df["log_price"] = (
    11.5
    + 0.0003 * df["sqft"]
    + 0.05   * df["bedrooms"]
    + 0.08   * df["bathrooms"]
    - 0.003  * df["age_years"]
    - 0.02   * df["dist_cbd_km"]
    + np.random.normal(0, 0.15, n)   # noise term
)

# ------------------------------------------------------------
# SECTION 2: FIT THE OLS MODEL VIA STATSMODELS
# ------------------------------------------------------------

feature_cols = ["sqft", "bedrooms", "bathrooms", "age_years", "dist_cbd_km"]

X = sm.add_constant(df[feature_cols])   # prepend intercept column (β₀)
y = df["log_price"]

model  = sm.OLS(y, X)                   # define the OLS specification
result = model.fit()                    # estimate β̂ via least squares

print(result.summary())                 # full regression table

# ------------------------------------------------------------
# SECTION 3: EXTRACT KEY DIAGNOSTICS FROM THE RESULTS OBJECT
# ------------------------------------------------------------

# result.fittedvalues → pandas Series of ŷᵢ (in-sample predictions)
# produced by X @ β̂ for every observation in the training set
y_hat = result.fittedvalues

# result.resid → pandas Series of eᵢ = yᵢ - ŷᵢ (OLS residuals)
# statsmodels computes this internally; we do NOT recompute manually
residuals = result.resid

# Compute RMSE from the residuals vector
rmse = np.sqrt(np.mean(residuals ** 2))
print(f"\nRMSE: {rmse:.4f}")

# ------------------------------------------------------------
# SECTION 4: FLAG OUTLIERS BEYOND ±2 STANDARD DEVIATIONS
# ------------------------------------------------------------

resid_std  = residuals.std()            # σ̂ of the residual distribution
resid_mean = residuals.mean()           # should be ≈ 0 for OLS by construction

# Boolean mask: True where |eᵢ - ē| > 2σ̂
outlier_mask = (residuals - resid_mean).abs() > 2 * resid_std

# Assign human-readable labels used by Plotly's color channel
# "Outlier (>2σ)" observations will render in crimson
point_labels = np.where(outlier_mask, "Outlier (>2σ)", "Normal")

# ------------------------------------------------------------
# SECTION 5: BUILD THE PLOTLY DASHBOARD
# ------------------------------------------------------------

plot_df = pd.DataFrame({
    "Fitted Values (ŷ)":  y_hat.values,
    "Residuals (e)":      residuals.values,
    "Status":             point_labels,
    # Observation index as hover info for traceability
    "Obs #":              np.arange(1, n + 1),
})

# Discrete color map: normal points in steel blue, outliers in crimson
COLOR_MAP = {
    "Normal":        "#4A90D9",   # steel blue
    "Outlier (>2σ)": "#C0392B",   # stark crimson
}

fig = px.scatter(
    plot_df,
    x="Fitted Values (ŷ)",
    y="Residuals (e)",
    color="Status",               # color channel driven by the outlier mask
    color_discrete_map=COLOR_MAP,
    hover_data=["Obs #"],         # expose observation index on hover
    title=(
        f"<b>Residual Forensics Dashboard</b>  —  "
        f"Hedonic OLS Model  |  RMSE = {rmse:.4f}"
    ),
    labels={
        "Fitted Values (ŷ)": "Fitted Values  ŷᵢ",
        "Residuals (e)":      "Residuals  eᵢ = yᵢ − ŷᵢ",
    },
    template="plotly_white",
    opacity=0.80,
)

# --- ZERO REFERENCE LINE ---
# add_hline is the non-deprecated replacement for layout.shapes dict syntax
fig.add_hline(
    y=0,
    line_width=1.8,
    line_dash="dash",
    line_color="#2C3E50",         # dark charcoal for contrast
    annotation_text=" E[e] = 0",
    annotation_position="bottom right",
    annotation_font_size=11,
)

# --- ±2σ BAND (shaded region for visual reference) ---
fig.add_hrect(
    y0=-2 * resid_std,
    y1= 2 * resid_std,
    fillcolor="rgba(74, 144, 217, 0.07)",  # translucent blue band
    line_width=0,
    annotation_text="±2σ band",
    annotation_position="top left",
    annotation_font_size=10,
    annotation_font_color="grey",
)

# --- LAYOUT POLISH ---
fig.update_traces(marker_size=7, marker_line_width=0.4, marker_line_color="white")
fig.update_layout(
    font_family="Inter, Arial, sans-serif",
    title_font_size=16,
    legend_title_text="Point Type",
    xaxis=dict(showgrid=True, gridcolor="#EEEEEE"),
    yaxis=dict(showgrid=True, gridcolor="#EEEEEE", zeroline=False),
    margin=dict(t=70, b=60, l=60, r=30),
    height=560,
)

fig.show()

                            OLS Regression Results                            
Dep. Variable:              log_price   R-squared:                       0.873
Model:                            OLS   Adj. R-squared:                  0.871
Method:                 Least Squares   F-statistic:                     404.1
Date:                Mon, 27 Apr 2026   Prob (F-statistic):          2.09e-129
Time:                        13:50:48   Log-Likelihood:                 160.32
No. Observations:                 300   AIC:                            -308.6
Df Residuals:                     294   BIC:                            -286.4
Df Model:                           5                                         
Covariance Type:            nonrobust                                         
                  coef    std err          t      P>|t|      [0.025      0.975]
-------------------------------------------------------------------------------
const          11.4725      0.040    286.846      